In [1]:
# CELL 1: Create Deployment Directory Structure
import os
import torch
import shutil
from pathlib import Path

# Create directories
os.makedirs("./hf_space", exist_ok=True)
os.makedirs("./hf_model", exist_ok=True)

print("✅ Created deployment directories")
print(f"   - ./hf_space/ (for Gradio Space)")
print(f"   - ./hf_model/ (for Model Hub)")

✅ Created deployment directories
   - ./hf_space/ (for Gradio Space)
   - ./hf_model/ (for Model Hub)


In [2]:
# CELL 2: Locate Your Trained Model
"""
Find and verify your trained model file
"""

# Common paths to check
possible_paths = [
    './models/model5_best_acc.pth',
    './models/Model5_EfficientNet_SegFormer_best_acc.pth',
    './model5_best_acc.pth',
    './best_model.pth'
]

model_path = None
for path in possible_paths:
    if os.path.exists(path):
        model_path = path
        break

if model_path:
    print(f"✅ Found trained model: {model_path}")
    size_mb = os.path.getsize(model_path) / 1024 / 1024
    print(f"   Model size: {size_mb:.2f} MB")
    
    # Load and inspect
    state_dict = torch.load(model_path, map_location='cpu')
    print(f"   Number of parameters: {len(state_dict.keys())}")
else:
    print("⚠️ No trained model found. Please update the path to your model.")
    print("   Creating dummy model for testing...")
    
    # We'll create a dummy model later

✅ Found trained model: ./models/model5_best_acc.pth
   Model size: 44.48 MB
   Number of parameters: 623


In [3]:
# CELL 3: Create model.py - The Architecture Definition
"""
Create a standalone model definition file that will be used by both the Space and Model Hub
"""

model_def = '''
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class EfficientNetEncoder(nn.Module):
    """EfficientNet-B3 Encoder with multi-scale features"""
    def __init__(self, model_name='efficientnet_b3', pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        self.feature_channels = self.backbone.feature_info.channels()
    
    def forward(self, x):
        return self.backbone(x)


class SegFormerDecoder(nn.Module):
    """SegFormer-style MLP decoder"""
    def __init__(self, encoder_channels, num_classes=1, decoder_dim=256):
        super().__init__()
        self.proj = nn.ModuleList()
        for in_ch in encoder_channels:
            self.proj.append(
                nn.Sequential(
                    nn.Conv2d(in_ch, decoder_dim, kernel_size=1),
                    nn.BatchNorm2d(decoder_dim),
                    nn.ReLU(inplace=True)
                )
            )
        self.fusion = nn.Sequential(
            nn.Conv2d(decoder_dim * len(encoder_channels), decoder_dim, kernel_size=1),
            nn.BatchNorm2d(decoder_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(decoder_dim, decoder_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(decoder_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(decoder_dim, num_classes, kernel_size=1)
        )
    
    def forward(self, features):
        target_size = features[0].shape[2]
        projected = []
        for i, feat in enumerate(features):
            proj = self.proj[i](feat)
            if proj.shape[2] != target_size:
                proj = F.interpolate(proj, size=(target_size, target_size), 
                                     mode='bilinear', align_corners=False)
            projected.append(proj)
        concat = torch.cat(projected, dim=1)
        out = self.fusion(concat)
        out = F.interpolate(out, size=(224, 224), mode='bilinear', align_corners=False)
        return out


class LungUltrasoundModel(nn.Module):
    """Complete multi-task model: EfficientNet + SegFormer"""
    def __init__(self, num_classes=3, num_seg_classes=1):
        super().__init__()
        self.encoder = EfficientNetEncoder('efficientnet_b3', pretrained=False)
        encoder_channels = self.encoder.feature_channels
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(encoder_channels[-1], 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
        # Segmentation decoder
        self.seg_decoder = SegFormerDecoder(encoder_channels, num_seg_classes, 256)
    
    def forward(self, x):
        features = self.encoder(x)
        class_out = self.classifier(features[-1])
        seg_out = self.seg_decoder(features)
        return class_out, seg_out
'''

# Save model.py
with open("./hf_space/model.py", "w", encoding='utf-8') as f:
    f.write(model_def)
print("✅ Created model.py in ./hf_space/")

# Also save for model hub
with open("./hf_model/model.py", "w", encoding='utf-8') as f:
    f.write(model_def)
print("✅ Created model.py in ./hf_model/")

✅ Created model.py in ./hf_space/
✅ Created model.py in ./hf_model/


In [4]:
# CELL 4: Create app.py - The Gradio Interface
"""
Create the main application file for Hugging Face Space
"""

app_content = '''
import gradio as gr
import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

# Import model definition
from model import LungUltrasoundModel

# Class names
CLASS_NAMES = ['COVID-19', 'Other Disease', 'Healthy']

# Device (use CPU for Hugging Face Spaces)
device = torch.device('cpu')
print(f"Using device: {device}")

# Load model
model_path = "pytorch_model.bin"
model = LungUltrasoundModel(num_classes=3, num_seg_classes=1)

if os.path.exists(model_path):
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()
    print(f"Model loaded from {model_path}")
else:
    print(f"Warning: Model file not found at {model_path}")
    print("Using random weights - predictions will be random")

def preprocess_image(image):
    """Preprocess image for model input"""
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    image = image.resize((224, 224))
    img_array = np.array(image).astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(img_array).permute(2, 0, 1).unsqueeze(0)
    return img_tensor

def create_overlay(original_image, seg_mask, alpha=0.5):
    """Create red overlay for B-line segmentation"""
    if isinstance(original_image, np.ndarray):
        img = original_image.copy()
    else:
        img = np.array(original_image)
    h, w = img.shape[:2]
    mask_resized = cv2.resize(seg_mask, (w, h))
    overlay = np.zeros_like(img)
    overlay[:, :, 0] = mask_resized * 255
    result = cv2.addWeighted(img, 1 - alpha, overlay, alpha, 0)
    return result

def predict(image):
    """Run inference on a single image"""
    img_tensor = preprocess_image(image).to(device)
    with torch.no_grad():
        logits, seg_logits = model(img_tensor)
        probs = torch.softmax(logits, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()
        confidence = probs[0, pred_class].item()
        seg_probs = torch.sigmoid(seg_logits[0, 0]).cpu().numpy()
        seg_mask = (seg_probs > 0.5).astype(np.float32)
    return CLASS_NAMES[pred_class], confidence, seg_mask, seg_probs

def gradio_predict(image):
    """Gradio interface prediction function"""
    class_name, confidence, seg_mask, seg_probs = predict(image)
    
    # Create overlay
    overlay = create_overlay(image, seg_mask)
    
    # Create probability heatmap
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(seg_probs, cmap='hot', vmin=0, vmax=1)
    ax.set_title('B-line Probability', fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    
    # Convert figure to image
    fig.canvas.draw()
    heatmap = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    heatmap = heatmap.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)
    
    # Format result text
    result = f"**Prediction:** {class_name}\\n**Confidence:** {confidence:.1%}"
    
    return overlay, heatmap, result

# Create Gradio interface
iface = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type="pil", label="Upload Lung Ultrasound Image"),
    outputs=[
        gr.Image(label="Segmentation Overlay (B-lines in red)"),
        gr.Image(label="B-line Probability Heatmap"),
        gr.Markdown(label="Results")
    ],
    title="Lung Ultrasound AI Assistant",
    description="""
    ## AI-Powered Lung Ultrasound Analysis
    
    Upload a lung ultrasound image to get:
    - **Disease classification**: COVID-19, Other Lung Disease, or Healthy
    - **B-line detection**: Segmentation overlay showing where B-lines are
    - **Probability heatmap**: Visual explanation of model confidence
    
    ### Instructions:
    1. Click to upload or drag an image
    2. Wait for analysis (1-2 seconds)
    3. View the results
    
    *Note: This is a research tool. Always consult a healthcare professional for diagnosis.*
    """,
    article="""
    <div style="text-align: center">
    <p><b>Citation:</b> Serunjogi, H., et al. "Edge-Optimized Multi-Task Deep Learning for Lung Ultrasound Analysis" (2026)</p>
    <p><b>Model:</b> EfficientNet-B3 + SegFormer | Trained on 1,463 images from Ugandan hospitals</p>
    </div>
    """
)

if __name__ == "__main__":
    iface.launch(server_name="0.0.0.0", server_port=7860)
'''

# Save app.py
with open("./hf_space/app.py", "w", encoding='utf-8') as f:
    f.write(app_content)
print("✅ Created app.py in ./hf_space/")

✅ Created app.py in ./hf_space/


In [5]:
# CELL 5: Create requirements.txt
"""
List all dependencies needed for the Space
"""

requirements = """gradio==3.50.0
torch==2.0.1
torchvision==0.15.2
timm==0.9.2
opencv-python==4.8.0
pillow==10.0.0
numpy==1.23.5
matplotlib==3.6.3
"""

with open("./hf_space/requirements.txt", "w", encoding='utf-8') as f:
    f.write(requirements)
print("✅ Created requirements.txt in ./hf_space/")

✅ Created requirements.txt in ./hf_space/


In [9]:
# CELL 6: Create Space README (FIXED)
"""
README for Hugging Face Space
"""

space_readme_lines = [
    "---",
    "title: Lung Ultrasound AI",
    "emoji: lungs",
    "colorFrom: blue",
    "colorTo: teal",
    "sdk: gradio",
    "sdk_version: 3.50.0",
    "app_file: app.py",
    "pinned: false",
    "---",
    "",
    "# Lung Ultrasound AI Assistant",
    "",
    "AI-powered lung ultrasound analysis for resource-constrained settings.",
    "",
    "## Features",
    "- Disease classification (COVID-19, Other Disease, Healthy)",
    "- B-line detection with segmentation overlay",
    "- Probability heatmap for explainability",
    "",
    "## Model Details",
    "- **Architecture**: EfficientNet-B3 + SegFormer",
    "- **Training Data**: 1,463 images from Mulago and Kiruddu Hospitals, Uganda",
    "- **Model Size**: 44 MB",
    "- **Inference Time**: <200 ms on CPU",
    "",
    "## Usage",
    "1. Upload a lung ultrasound image",
    "2. Wait for analysis",
    "3. View classification and B-line detection results",
    "",
    "## Citation",
    "```bibtex",
    "@article{serunjogi2026lung,",
    "  title={Edge-Optimized Multi-Task Deep Learning for Lung Ultrasound Analysis},",
    "  author={Serunjogi Huzaifa},",
    "  journal={IEEE Access},",
    "  year={2026}",
    "}",
    "```"
]

with open("./hf_space/README.md", "w", encoding='utf-8') as f:
    f.write('\n'.join(space_readme_lines))
print("✅ Created README.md in ./hf_space/")

✅ Created README.md in ./hf_space/


In [10]:
# CELL 7: Copy Model Weights to Space (FIXED)
"""
Copy the trained model weights to the Space directory
"""

import os
import shutil
import torch

# Update this path to your actual model file
TRAINED_MODEL_PATH = "./models/model5_best_acc.pth"  # Change this!
SPACE_MODEL_PATH = "./hf_space/pytorch_model.bin"

# Check common model locations
possible_paths = [
    TRAINED_MODEL_PATH,
    "./models/Model5_EfficientNet_SegFormer_best_acc.pth",
    "./models/model5_best_acc.pth",
    "./model5_best_acc.pth",
    "./best_model.pth"
]

model_found = False
for path in possible_paths:
    if os.path.exists(path):
        TRAINED_MODEL_PATH = path
        model_found = True
        break

if model_found:
    shutil.copy(TRAINED_MODEL_PATH, SPACE_MODEL_PATH)
    print(f"✅ Copied model from {TRAINED_MODEL_PATH} to {SPACE_MODEL_PATH}")
    
    # Verify
    size_mb = os.path.getsize(SPACE_MODEL_PATH) / 1024 / 1024
    print(f"   Model size: {size_mb:.2f} MB")
    
    # Test load
    try:
        state_dict = torch.load(SPACE_MODEL_PATH, map_location='cpu')
        print(f"   Verified: {len(state_dict.keys())} parameters")
    except Exception as e:
        print(f"   Warning: Could not verify model - {e}")
else:
    print(f"⚠️ Model not found. Searched paths:")
    for path in possible_paths:
        print(f"   - {path}")
    print("\n   Creating dummy model for testing...")
    
    # Create dummy model
    try:
        # First, we need to import the model definition
        # Create a temporary model.py if it doesn't exist
        if not os.path.exists("./hf_space/model.py"):
            print("   Creating model.py first...")
            # This will be created in the next cell
        
        # For now, just note that model needs to be created
        print("   Please ensure model.py is created before running this cell")
        print("   Run the previous cells first to create model.py")
    except:
        pass

✅ Copied model from ./models/model5_best_acc.pth to ./hf_space/pytorch_model.bin
   Model size: 44.48 MB
   Verified: 623 parameters


In [11]:
# CELL 8: Create Model Hub README (FIXED)
"""
Prepare files for Hugging Face Model Hub
"""

# Create model hub README
model_readme_lines = [
    "---",
    "language: en",
    "license: mit",
    "tags:",
    "- medical",
    "- ultrasound",
    "- lung",
    "- segmentation",
    "- classification",
    "---",
    "",
    "# Lung Ultrasound AI Model",
    "",
    "## Model Description",
    "Multi-task model for lung ultrasound analysis:",
    "- Classification: COVID-19, Other Lung Disease, Healthy",
    "- Segmentation: B-line detection and localization",
    "",
    "## Architecture",
    "- Encoder: EfficientNet-B3",
    "- Decoder: SegFormer-style MLP",
    "- Multi-task: Hard parameter sharing",
    "",
    "## Training Data",
    "- Classification: 1,062 images from Ugandan hospitals",
    "- Segmentation: 401 images with B-line annotations",
    "",
    "## Performance",
    "| Metric | Value |",
    "|--------|-------|",
    "| Accuracy | 23.40% |",
    "| Precision | 39.96% |",
    "| Model Size | 44 MB |",
    "",
    "## Usage",
    "```python",
    "import torch",
    "from model import LungUltrasoundModel",
    "",
    "model = LungUltrasoundModel()",
    "model.load_state_dict(torch.load('pytorch_model.bin'))",
    "model.eval()",
    "",
    "# Preprocess image (224x224 RGB, normalized to 0-1)",
    "# Run inference",
    "class_logits, seg_logits = model(image_tensor)",
    "```"
]

with open("./hf_model/README.md", "w", encoding='utf-8') as f:
    f.write('\n'.join(model_readme_lines))
print("✅ Created README.md in ./hf_model/")

✅ Created README.md in ./hf_model/


In [12]:
# CELL 9: Create ZIP Packages (FIXED)
"""
Create ZIP files for easy upload to Hugging Face
"""

import zipfile
import os

def create_zip(source_dir, output_name):
    """Create a ZIP file from a directory"""
    zip_path = f"{output_name}.zip"
    
    # Check if directory exists
    if not os.path.exists(source_dir):
        print(f"⚠️ Directory not found: {source_dir}")
        return None
    
    # Get list of files
    files = []
    for root, dirs, filenames in os.walk(source_dir):
        for filename in filenames:
            files.append(os.path.join(root, filename))
    
    if not files:
        print(f"⚠️ No files in {source_dir}")
        return None
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in files:
            arcname = os.path.relpath(file_path, source_dir)
            zipf.write(file_path, arcname)
    
    # Show contents
    print(f"\nContents of {zip_path}:")
    with zipfile.ZipFile(zip_path, 'r') as zipf:
        for name in zipf.namelist():
            info = zipf.getinfo(name)
            size_kb = info.file_size / 1024
            if size_kb > 1024:
                print(f"   - {name} ({size_kb/1024:.2f} MB)")
            else:
                print(f"   - {name} ({size_kb:.2f} KB)")
    
    return zip_path

# Create Space package
print("Creating Space package...")
space_zip = create_zip("./hf_space", "hf_space_upload")

# Create Model Hub package
print("\nCreating Model Hub package...")
model_zip = create_zip("./hf_model", "hf_model_upload")

print("\n" + "="*60)
print("DEPLOYMENT PACKAGES READY")
print("="*60)
if space_zip:
    print(f"Space ZIP: {space_zip}")
if model_zip:
    print(f"Model ZIP: {model_zip}")

Creating Space package...

Contents of hf_space_upload.zip:
   - app.py (4.53 KB)
   - model.py (3.13 KB)
   - pytorch_model.bin (44.48 MB)
   - README.md (0.94 KB)
   - requirements.txt (0.13 KB)

Creating Model Hub package...

Contents of hf_model_upload.zip:
   - config.json (0.61 KB)
   - model.py (3.13 KB)
   - pytorch_model.bin (44.48 MB)
   - README.md (0.97 KB)

DEPLOYMENT PACKAGES READY
Space ZIP: hf_space_upload.zip
Model ZIP: hf_model_upload.zip


In [ ]:
# CELL 10: Final Verification (FIXED)
"""
Check that all required files exist
"""

import os

required_files = [
    "./hf_space/app.py",
    "./hf_space/model.py",
    "./hf_space/requirements.txt",
    "./hf_space/README.md",
    "./hf_space/pytorch_model.bin",
    "./hf_model/model.py",
    "./hf_model/pytorch_model.bin",
    "./hf_model/README.md"
]

print("="*60)
print("VERIFYING DEPLOYMENT FILES")
print("="*60)

all_ok = True
for file in required_files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024
        if size > 1024:
            print(f"✅ {file} ({size/1024:.2f} MB)")
        else:
            print(f"✅ {file} ({size:.2f} KB)")
    else:
        print(f"❌ {file} - MISSING")
        all_ok = False



VERIFYING DEPLOYMENT FILES
✅ ./hf_space/app.py (4.53 KB)
✅ ./hf_space/model.py (3.13 KB)
✅ ./hf_space/requirements.txt (0.13 KB)
✅ ./hf_space/README.md (0.94 KB)
✅ ./hf_space/pytorch_model.bin (44.48 MB)
✅ ./hf_model/model.py (3.13 KB)
✅ ./hf_model/pytorch_model.bin (44.48 MB)
✅ ./hf_model/README.md (0.97 KB)

✅ ALL FILES READY FOR DEPLOYMENT!

Next steps:
1. Go to https://huggingface.co/
2. Create a new Model Repository and upload hf_model_upload.zip contents
3. Create a new Space and upload hf_space_upload.zip contents
4. Your model will be live in 5-10 minutes


In [14]:
# Run this to see what files you have
import os

print("Space files (./hf_space/):")
for f in os.listdir("./hf_space"):
    size = os.path.getsize(f"./hf_space/{f}") / 1024
    if size > 1024:
        print(f"  - {f} ({size/1024:.2f} MB)")
    else:
        print(f"  - {f} ({size:.2f} KB)")

print("\nModel files (./hf_model/):")
for f in os.listdir("./hf_model"):
    size = os.path.getsize(f"./hf_model/{f}") / 1024
    if size > 1024:
        print(f"  - {f} ({size/1024:.2f} MB)")
    else:
        print(f"  - {f} ({size:.2f} KB)")

Space files (./hf_space/):
  - app.py (4.53 KB)
  - model.py (3.13 KB)
  - pytorch_model.bin (44.48 MB)
  - README.md (0.94 KB)
  - requirements.txt (0.13 KB)

Model files (./hf_model/):
  - config.json (0.61 KB)
  - model.py (3.13 KB)
  - pytorch_model.bin (44.48 MB)
  - README.md (0.97 KB)
